In [1]:
import torch
import torch.nn as nn
import math
import numpy as np
import pandas as pd
import psutil
import time
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import gc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [2]:
import math
import torch
import torch.nn as nn

class MixedDimensionEmbedding(nn.Module):
    def __init__(self, block_vocab_sizes, base_dim, alpha=0.5):
        super(MixedDimensionEmbedding, self).__init__()
        self.num_blocks = len(block_vocab_sizes)
        self.base_dim = base_dim
        
        self.embeddings = nn.ModuleList()
        self.projections = nn.ModuleList()
        
        v_min = float(min(block_vocab_sizes))
        
        for vocab_size in block_vocab_sizes:
            # Tính toán số chiều động
            raw_dim = base_dim * ((v_min / float(vocab_size)) ** alpha)
            dim = max(1, 2 ** round(math.log2(max(raw_dim, 1e-9))))
            dim = min(dim, base_dim)
            emb_dim = int(dim)
            
            # Khởi tạo lớp nhúng với số chiều tối ưu
            self.embeddings.append(nn.Embedding(vocab_size, emb_dim))
            
            # Lớp Linear để đồng bộ số chiều về base_dim
            if emb_dim != base_dim:
                self.projections.append(nn.Linear(emb_dim, base_dim, bias=False))
            else:
                self.projections.append(nn.Identity())
                
    def forward(self, cat_x):
        projected_embs = []
        for i in range(self.num_blocks):
            feature_col = cat_x[:, i] 
            emb = self.embeddings[i](feature_col)
            proj = self.projections[i](emb)
            projected_embs.append(proj)
            
        # Ở DeepFM, ta cần Tensor 3D để tính FM bậc 2 nên dùng STACK thay vì CAT
        # Shape trả về: (batch_size, num_blocks, base_dim)
        return torch.stack(projected_embs, dim=1)

In [3]:
class DeepFMMDE(nn.Module):
    def __init__(self, num_dense_features, vocab_sizes, base_dim=64, alpha=0.5, hidden_dims=[256, 128, 64]):
        super(DeepFMMDE, self).__init__()
        self.num_sparse_features = len(vocab_sizes)
        self.num_dense_features = num_dense_features
        
        # 1. Thành phần FM bậc 1 (Linear)
        # Bậc 1 cho Sparse (vẫn dùng Embedding 1 chiều riêng biệt cho chuẩn kiến trúc)
        self.fm_1st_sparse = nn.ModuleList([nn.Embedding(vocab, 1) for vocab in vocab_sizes])
        
        # Bậc 1 cho Dense
        if num_dense_features > 0:
            self.fm_1st_dense = nn.Linear(num_dense_features, 1)
        
        # 2. Thành phần FM bậc 2 và Đầu vào cho Deep (Dùng MDE)
        self.mde = MixedDimensionEmbedding(
            block_vocab_sizes=vocab_sizes, 
            base_dim=base_dim, 
            alpha=alpha
        )
        
        # 3. Thành phần Deep
        # Đầu vào của Deep là MDE đã được trải phẳng (flatten) + Dense features
        deep_in_dim = (self.num_sparse_features * base_dim) + num_dense_features
        deep_layers = []
        in_dim = deep_in_dim
        for dim in hidden_dims:
            deep_layers.append(nn.Linear(in_dim, dim))
            deep_layers.append(nn.ReLU())
            in_dim = dim
        deep_layers.append(nn.Linear(in_dim, 1))
        self.deep_net = nn.Sequential(*deep_layers)

    def forward(self, dense_x, sparse_x):
        # --- 1. FM: Tương tác bậc 1 ---
        # Lấy từng item trong sparse_x đưa qua embedding 1D, rồi cộng tổng lại
        fm_1st = sum([self.fm_1st_sparse[i](sparse_x[:, i].unsqueeze(1)) for i in range(self.num_sparse_features)])
        fm_1st = fm_1st.squeeze(1) # Đảm bảo shape là (batch_size, 1)
        
        if self.num_dense_features > 0:
            fm_1st = fm_1st + self.fm_1st_dense(dense_x)
        
        # --- 2. Lấy Embeddings chung (Qua lớp MDE) ---
        # emb_tensor có shape: (batch_size, num_sparse_features, base_dim)
        emb_tensor = self.mde(sparse_x) 
        
        # --- 3. FM: Tương tác bậc 2 ---
        sum_of_emb = torch.sum(emb_tensor, dim=1) # (batch_size, base_dim)
        sum_of_emb_sq = sum_of_emb ** 2
        
        sq_of_emb = emb_tensor ** 2
        sum_of_sq_emb = torch.sum(sq_of_emb, dim=1) # (batch_size, base_dim)
        
        fm_2nd = 0.5 * torch.sum(sum_of_emb_sq - sum_of_sq_emb, dim=1, keepdim=True) # (batch_size, 1)
        
        # --- 4. Deep Component ---
        # Trải phẳng emb_tensor từ 3D xuống 2D
        deep_in = emb_tensor.view(sparse_x.size(0), -1) 
        if self.num_dense_features > 0:
            deep_in = torch.cat([dense_x, deep_in], dim=1) 
            
        deep_out = self.deep_net(deep_in) # (batch_size, 1)
        
        # Tổng hợp LOGITS và ép về 1D
        out = fm_1st + fm_2nd + deep_out
        return out.squeeze(1)

In [4]:
import math
from torch.utils.data import Dataset
import torch
from tqdm.auto import tqdm

class StandardCriteoDataset(Dataset):
    def __init__(self, hf_dataset, dense_cols, cat_cols, cat_mappers=None, block_vocab_sizes=None):
        self.data = hf_dataset
        self.dense_cols = dense_cols
        self.cat_cols = cat_cols
        
        # Nếu không truyền vào mappers (tức là đang nạp tập Train) -> Tự xây dựng
        if cat_mappers is None or block_vocab_sizes is None:
            print("Đang đếm Vocabulary Size cho tập TRAIN...")
            self.block_vocab_sizes = []
            self.cat_mappers = []
            self._build_vocab()
        else:
            # Nếu đã có mappers (Tập Val/Test) -> Dùng lại của Train để chống Leakage
            print("Sử dụng Vocabulary đã học từ tập TRAIN...")
            self.cat_mappers = cat_mappers
            self.block_vocab_sizes = block_vocab_sizes
        
    def _build_vocab(self):
        df = self.data.to_pandas()
        for col in tqdm(self.cat_cols, desc="Building Vocab"):
            unique_vals = df[col].dropna().unique()
            mapper = {val: idx + 1 for idx, val in enumerate(unique_vals)}
            self.cat_mappers.append(mapper)
            # Kích thước = số unique + 1 (chừa index 0 cho Padding/OOV)
            self.block_vocab_sizes.append(len(unique_vals) + 1)
            
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        
        dense_x = []
        for c in self.dense_cols:
            val = row[c]
            if val is None or val < 0:
                dense_x.append(0.0)
            else:
                dense_x.append(math.log1p(float(val))) 
        
        # Categorical
        cat_ids = []
        for col_idx, col_name in enumerate(self.cat_cols):
            val = row[col_name]
            mapper = self.cat_mappers[col_idx]
            # ID nào chưa từng thấy ở Train -> gán = 0 (OOV)
            cat_ids.append(mapper.get(val, 0)) 
            
        label = float(row['label'])

        return (
            torch.tensor(dense_x, dtype=torch.float32),
            torch.tensor(cat_ids, dtype=torch.long),
            torch.tensor(label, dtype=torch.float32)
        )

In [5]:
from datasets import load_dataset
from torch.utils.data import DataLoader
import os

dense_cols = [f"I{i}" for i in range(1, 14)]
cat_cols = [f"C{i}" for i in range(1, 27)]

train_file = "/kaggle/input/datasets/nmpogg/criteo-cleaned-v2/train.parquet"
val_file = "/kaggle/input/datasets/nmpogg/criteo-cleaned-v2/validation.parquet"

train_hf = load_dataset("parquet", data_files=train_file, split="train")
val_hf = load_dataset("parquet", data_files=val_file, split="train")

print(f"Số mẫu -> Train: {len(train_hf)} | Val: {len(val_hf)}")

print("Đang khởi tạo Dataset (Xây dựng Vocab trên Train)...")
train_dataset = StandardCriteoDataset(train_hf, dense_cols, cat_cols)

# Lấy từ điển đã học
train_mappers = train_dataset.cat_mappers
train_vocab_sizes = train_dataset.block_vocab_sizes

print("Đang khởi tạo Val Dataset...")
val_dataset = StandardCriteoDataset(val_hf, dense_cols, cat_cols, cat_mappers=train_mappers, block_vocab_sizes=train_vocab_sizes)

print("Đang tạo DataLoader...")
train_loader = DataLoader(train_dataset, batch_size=8192, shuffle=True, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=2)
val_loader = DataLoader(val_dataset, batch_size=8192, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=2)

print("Hoàn tất tạo DataLoader!")

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/32 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Số mẫu -> Train: 36665662 | Val: 4583562
Đang khởi tạo Dataset (Xây dựng Vocab trên Train)...
Đang đếm Vocabulary Size cho tập TRAIN...


Building Vocab:   0%|          | 0/26 [00:00<?, ?it/s]

Đang khởi tạo Val Dataset...
Sử dụng Vocabulary đã học từ tập TRAIN...
Đang tạo DataLoader...
Hoàn tất tạo DataLoader!


In [6]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()
print(f"Số GPU khả dụng: {num_gpus}")
for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

# Khởi tạo mô hình DeepFM
model = DeepFMMDE(
    num_dense_features=len(dense_cols),
    vocab_sizes=train_vocab_sizes,
    base_dim=64,                # MDE sẽ đưa mọi embedding về số chiều cơ sở này
    alpha=0.5,                  # Tham số nội suy kích thước (0.0 đến 1.0)
    hidden_dims=[256, 128, 64]  # Mạng Deep
)

# Wrap DataParallel nếu có >= 2 GPU
if num_gpus > 1:
    print(f"Dùng DataParallel trên {num_gpus} GPU")
    model = nn.DataParallel(model)

model = model.to(device)

# Vì đã bỏ sigmoid trong DeepFM, dùng BCEWithLogitsLoss là hoàn toàn chính xác
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)

Số GPU khả dụng: 1
  GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [7]:
import torch
from torch.utils.flop_counter import FlopCounterMode

print(f"{'Tên Component / Lớp (Layer)':<45} | {'Số lượng Params':<15}")
print("-" * 65)

total_params = 0

for name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue
    params = parameter.numel()
    clean_name = name.replace(".weight", "").replace(".bias", " (bias)")
    print(f"{clean_name:<45} | {params:,}")
    total_params += params

print("-" * 65)
print(f"{'TỔNG CỘNG THAM SỐ HUẤN LUYỆN:':<45} | {total_params:,}")
print("="*65)


print("\nĐang phân tích số lượng phép tính (FLOPs)...")
model.eval()

dummy_dense = torch.randn(1, len(dense_cols)).to(device)

dummy_cat = torch.zeros(1, len(cat_cols), dtype=torch.long).to(device)

flop_counter = FlopCounterMode(model, display=True)
with flop_counter:
    _ = model(dummy_dense, dummy_cat)

Tên Component / Lớp (Layer)                   | Số lượng Params
-----------------------------------------------------------------
fm_1st_sparse.0                               | 1,461
fm_1st_sparse.1                               | 578
fm_1st_sparse.2                               | 8,378,274
fm_1st_sparse.3                               | 1,884,136
fm_1st_sparse.4                               | 306
fm_1st_sparse.5                               | 25
fm_1st_sparse.6                               | 12,495
fm_1st_sparse.7                               | 634
fm_1st_sparse.8                               | 4
fm_1st_sparse.9                               | 88,846
fm_1st_sparse.10                              | 5,655
fm_1st_sparse.11                              | 6,950,098
fm_1st_sparse.12                              | 3,195
fm_1st_sparse.13                              | 28
fm_1st_sparse.14                              | 14,756
fm_1st_sparse.15                              | 4,589,441
fm_

/tmp/ipykernel_163/2912037424.py:29: UserWarning: mods argument is not needed anymore, you can stop passing it
  flop_counter = FlopCounterMode(model, display=True)


Module                       FLOP    % Total
-----------------------  --------  ---------
DeepFMMDE                968.090K    100.00%
 - aten.addmm            940.698K     97.17%
 - aten.mm                27.392K      2.83%
 DeepFMMDE.deep_net      940.672K     97.17%
  - aten.addmm           940.672K     97.17%
 DeepFMMDE.fm_1st_dense    0.026K      0.00%
  - aten.addmm             0.026K      0.00%
 DeepFMMDE.mde            27.392K      2.83%
  - aten.mm               27.392K      2.83%


In [12]:
print("Bắt đầu huấn luyện MDE-DCN...")
EPOCHS = 5
best_val_auc = 0.0
save_path = "best_mde_deepfm_model.pth"

torch.cuda.reset_peak_memory_stats()

for epoch in range(EPOCHS):
    # ---------- TRAIN ----------
    model.train()
    train_loss = 0.0
    total_ram_gb = 0.0
    total_vram_gb = 0.0
    batch_count = 0

    train_bar = tqdm(train_loader, desc=f"[TRAIN] Epoch {epoch+1}/{EPOCHS}")
    for dense_x, cat_x, labels in train_bar:
        dense_x = dense_x.to(device, non_blocking=True)
        cat_x   = cat_x.to(device, non_blocking=True)
        labels  = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(dense_x, cat_x)
        loss = criterion(logits, labels)
        
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        # Đo lường tài nguyên
        current_ram = psutil.virtual_memory().used / (1024**3)
        current_vram = torch.cuda.memory_allocated() / (1024**3)
        total_ram_gb += current_ram
        total_vram_gb += current_vram
        batch_count += 1

        train_bar.set_postfix({"Loss": f"{loss.item():.4f}"})

    avg_train_loss = train_loss / len(train_loader)

    # ---------- VALIDATION ----------
    model.eval()
    val_loss = 0.0
    val_targets, val_preds = [], []

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"[VAL] Epoch {epoch+1}/{EPOCHS}", leave=False)
        for dense_x, cat_x, labels in val_bar:
            dense_x = dense_x.to(device, non_blocking=True)
            cat_x   = cat_x.to(device, non_blocking=True)
            labels  = labels.to(device, non_blocking=True)

            logits = model(dense_x, cat_x)
            loss   = criterion(logits, labels)
            val_loss += loss.item()

            probs = torch.sigmoid(logits)
            
            # Lưu nguyên block mảng numpy
            val_targets.append(labels.cpu().numpy())
            val_preds.append(probs.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    
    # Gộp block numpy lại
    val_targets_arr = np.concatenate(val_targets)
    val_preds_arr = np.concatenate(val_preds)
    val_auc = roc_auc_score(val_targets_arr, val_preds_arr)

    # Thống kê
    avg_ram = total_ram_gb / batch_count
    avg_vram = total_vram_gb / batch_count
    max_vram = torch.cuda.max_memory_allocated() / (1024**3)

    print(f"Tài nguyên Epoch {epoch+1}:")
    print(f"- RAM trung bình:  {avg_ram:.2f} GB")
    print(f"- VRAM trung bình: {avg_vram:.2f} GB | Đỉnh điểm: {max_vram:.2f} GB")

    print(
        f"Epoch {epoch+1}: \n"
        f"Train Loss: {avg_train_loss:.4f} || "
        f"Val Loss: {avg_val_loss:.4f} | Val AUC: {val_auc:.4f}\n"
    )

    if val_auc > best_val_auc:
        print(f"Best Val AUC: {val_auc:.4f}. Đang lưu model...")
        best_val_auc = val_auc
        torch.save(model.state_dict(), save_path)
    else:
        print("Val AUC không cải thiện.")
    
    # Dọn rác
    del val_targets, val_preds, val_targets_arr, val_preds_arr
    gc.collect()

Bắt đầu huấn luyện MDE-DCN...


[TRAIN] Epoch 1/5:   0%|          | 0/4476 [00:01<?, ?it/s]

[VAL] Epoch 1/5:   0%|          | 0/560 [00:00<?, ?it/s]

Tài nguyên Epoch 1:
- RAM trung bình:  28.76 GB
- VRAM trung bình: 0.87 GB | Đỉnh điểm: 1.09 GB
Epoch 1: 
Train Loss: 0.4407 || Val Loss: 0.4823 | Val AUC: 0.7769

Best Val AUC: 0.7769. Đang lưu model...


[TRAIN] Epoch 2/5:   0%|          | 0/4476 [00:01<?, ?it/s]

...

[VAL] Epoch 4/5:   0%|          | 0/560 [00:00<?, ?it/s]

Tài nguyên Epoch 4:
- RAM trung bình:  28.80 GB
- VRAM trung bình: 0.87 GB | Đỉnh điểm: 1.09 GB
Epoch 4: 
Train Loss: 0.3645 || Val Loss: 0.5224 | Val AUC: 0.7494

Val AUC không cải thiện.


[TRAIN] Epoch 5/5:   0%|          | 0/4476 [00:01<?, ?it/s]

[VAL] Epoch 5/5:   0%|          | 0/560 [00:00<?, ?it/s]

Tài nguyên Epoch 5:
- RAM trung bình:  28.81 GB
- VRAM trung bình: 0.87 GB | Đỉnh điểm: 1.09 GB
Epoch 5: 
Train Loss: 0.3586 || Val Loss: 0.5269 | Val AUC: 0.7472

Val AUC không cải thiện.


In [ ]:
print("Bắt đầu đánh giá trên tập TEST và đo Latency...")

test_file = "/kaggle/input/datasets/nmpogg/criteo-cleaned-v2/test.parquet"
test_hf = load_dataset("parquet", data_files=test_file, split="train")
test_dataset = StandardCriteoDataset(test_hf, dense_cols, cat_cols, cat_mappers=train_mappers, block_vocab_sizes=train_vocab_sizes)
test_loader = DataLoader(
    test_dataset, 
    batch_size=8192, 
    shuffle=False, 
    num_workers=8, 
    pin_memory=True, 
    persistent_workers=True, 
    prefetch_factor=2
)

model.load_state_dict(torch.load("best_mde_deepfm_model.pth", map_location=device))
model.eval()

start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

total_latency_ms = 0.0
total_samples = 0
latency_batch_count = 0
WARMUP_BATCHES = 5 

test_loss = 0.0
test_targets, test_preds = [], []

with torch.no_grad():
    test_bar = tqdm(test_loader, desc="[TEST] Evaluating")
    for i, (dense_x, cat_x, labels) in enumerate(test_bar):
        dense_x = dense_x.to(device, non_blocking=True)
        cat_x   = cat_x.to(device, non_blocking=True)
        labels  = labels.to(device, non_blocking=True)

        batch_size = dense_x.size(0)

        # Bấm giờ Forward Pass
        start_event.record()
        logits = model(dense_x, cat_x)
        end_event.record()
        
        loss = criterion(logits, labels)
        test_loss += loss.item()

        probs = torch.sigmoid(logits)
        test_targets.append(labels.cpu().numpy())
        test_preds.append(probs.cpu().numpy())
        
        # Đồng bộ và tính toán thời gian
        torch.cuda.synchronize()
        if i >= WARMUP_BATCHES:
            batch_latency = start_event.elapsed_time(end_event)
            total_latency_ms += batch_latency
            total_samples += batch_size
            latency_batch_count += 1

avg_test_loss = test_loss / len(test_loader)
test_targets_arr = np.concatenate(test_targets)
test_preds_arr = np.concatenate(test_preds)
test_auc = roc_auc_score(test_targets_arr, test_preds_arr)

if latency_batch_count > 0:
    avg_batch_latency = total_latency_ms / latency_batch_count
    avg_inference_per_sample = total_latency_ms / total_samples
else:
    avg_batch_latency = 0; avg_inference_per_sample = 0

print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test AUC:  {test_auc:.4f}")

print(f"Latency 1 Batch (Size {test_loader.batch_size}): {avg_batch_latency:.2f} ms")
print(f"Latency 1 Dòng (Per Sample):      {avg_inference_per_sample:.4f} ms")

Bắt đầu đánh giá trên tập TEST và đo Latency...


Generating train split: 0 examples [00:00, ? examples/s]

Sử dụng Vocabulary đã học từ tập TRAIN...


[TEST] Evaluating:   0%|          | 0/561 [00:00<?, ?it/s]

Test Loss: 0.4815
Test AUC:  0.7772
Latency 1 Batch (Size 8192): 1.35 ms
Latency 1 Dòng (Per Sample):      0.0002 ms
